Cargamos los datos y los analizamos

In [12]:
import pandas as pd
import numpy as np
SEED = 42

In [13]:
business_data = pd.read_csv("./data/negocios.csv")
user_data = pd.read_csv("./data/usuarios.csv")

train_reviews = pd.read_csv("./data/train_reviews.csv")
test_reviews = pd.read_csv("./data/test_reviews.csv")


C:\Users\pable\AppData\Local\Temp\ipykernel_2732\824715385.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  user_data = pd.read_csv("./data/usuarios.csv")


In [14]:
# Convert to sets for fast lookup
user_set = set(user_data['user_id'])
business_set = set(business_data['business_id'])

# Unique IDs from test set
unique_test_users = test_reviews['user_id'].unique()
unique_test_business = test_reviews['business_id'].unique()

# Efficient check
missing_users = [user for user in unique_test_users if user not in user_set]
missing_businesses = [biz for biz in unique_test_business if biz not in business_set]

for user in missing_users:
    print(f"usuario {user} de test no encontrado en la base de datos")

for negocio in missing_businesses:
    print(f"negocio {negocio} de test no encontrado en la base de datos")

usuario I6G8wz_LD_8IsCPxvjs8SQ de test no encontrado en la base de datos


In [15]:
# observamos si existe algun usuario o negocio en test o train que no tengamos en la base de datos

unique_test_users = test_reviews['user_id'].unique() 
unique_test_business = test_reviews['business_id'].unique() 

# Efficient check
missing_users = [user for user in unique_test_users if user not in user_set]
missing_businesses = [biz for biz in unique_test_business if biz not in business_set]

for user in missing_users:
    print(f"usuario {user} de test no encontrado en la base de datos")

for negocio in missing_businesses:
    print(f"negocio {negocio} de test no encontrado en la base de datos")

unique_train_users = train_reviews['user_id'].unique() 
unique_train_business = train_reviews['business_id'].unique() 

# Efficient check
missing_users = [user for user in unique_train_users if user not in user_set]
missing_businesses = [biz for biz in unique_train_business if biz not in business_set]

for user in missing_users:
    print(f"usuario {user} de train no encontrado en la base de datos")

for negocio in missing_businesses:
    print(f"negocio {negocio} de train no encontrado en la base de datos")



usuario I6G8wz_LD_8IsCPxvjs8SQ de test no encontrado en la base de datos
usuario ufZfni7nb_KdJC6DXNfVHQ de train no encontrado en la base de datos


In [16]:
# Observamos de cuantas reviews desconocemos el usuario

print(train_reviews[train_reviews['user_id'] == 'ufZfni7nb_KdJC6DXNfVHQ']['review_id'].count())
print(test_reviews[test_reviews['user_id'] == 'I6G8wz_LD_8IsCPxvjs8SQ']['review_id'].count())


1
1


Dado que apenas existen reviews con usuarios que desconocemos, ignoramos el dato faltante.

## Enfoque utilizando clasificadores de ML

Vamos a intentar resolver este problema mediante clasificadores de machine learning, sin usar embeddings, redes neuronales ni algoritmos especializados en recomendaciones

In [17]:
# primero fusionamos los datos de reviews usuarios y negocios en un unico dataset

full_train_reviews = pd.merge(train_reviews, business_data, on='business_id', how='left',suffixes=('_review', '_business'))
full_train_reviews = pd.merge(full_train_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = pd.merge(test_reviews, business_data, on='business_id', how='left',suffixes=('_review', '_business'))
full_test_reviews = pd.merge(full_test_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = full_test_reviews.rename(columns={"stars": "stars_business"})
full_test_reviews.head()

,review_id,user_id,business_id,useful,funny,cool,text,date,name,address,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,Backatown Coffee Parlour,"301 Basin St, Ste 1",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,Lincoln Square Pancake House,5024 E 56th St,...,6.0,1.0,0.0,3.0,16.0,28.0,32.0,32.0,18.0,1.0
2,seR2KhblYMWg-k9zzN6aYA,hF68a0mpu97u0oaryFYhyg,_RG4IByyBR528CMc7DefJA,2,0,0,Came here when my kitten got very sick by the ...,2015-09-06 15:29:02,BluePearl Pet Hospital,301 Veterans Hwy,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,BToo00Fi5pfJFA5MI2HM5g,G4yX5Q1tFfwSucFOmiyjdA,xxlbRiWWQkk-6LST3Hd12g,2,0,0,So I'll preface by saying we did have an overa...,2015-09-14 00:49:17,The Backyard,244 W Harrison Ave,...,4.0,0.0,0.0,0.0,11.0,6.0,10.0,10.0,6.0,1.0
4,FHJAzi1imodBit3RWK7zQA,Srqi1xb7exdB9uRHxDeEkw,LgGqdFLD7-ca0Z9F_q4Fuw,0,0,0,This place is a joke. Worst bar service ever. ...,2015-07-24 01:03:40,Postcard Inn on the Beach,6300 Gulf Blvd,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


Seleccionamos aquellas columnas que nuestro modelo pueda utilizar sin realizar ningun preprocesamiento complejo

In [18]:
# vamos a utilizar como primer enfoque unicamente los valores de aquellas columnas que tan solo tengan un dato numerico y no valores agregados, esto nos permite entrenar un modelo de manera sencilla

learnable_columns = ['useful','funny','cool','fans','average_stars','compliment_hot','compliment_more','compliment_profile','compliment_cute','compliment_list',
                     'compliment_note','compliment_plain','compliment_cool','compliment_funny','compliment_writer','compliment_photos','latitude','longitude',
                     'stars_business','review_count','review_count_user','is_open','useful_user','funny_user','cool_user']

print(len(learnable_columns))
full_train_reviews.columns

25


Index(['review_id', 'user_id', 'business_id', 'stars_review', 'useful',
       'funny', 'cool', 'text', 'date', 'name', 'address', 'city', 'state',
       'postal_code', 'latitude', 'longitude', 'stars_business',
       'review_count', 'is_open', 'attributes', 'categories', 'hours',
       'name_user', 'review_count_user', 'yelping_since', 'useful_user',
       'funny_user', 'cool_user', 'elite', 'friends', 'fans', 'average_stars',
       'compliment_hot', 'compliment_more', 'compliment_profile',
       'compliment_cute', 'compliment_list', 'compliment_note',
       'compliment_plain', 'compliment_cool', 'compliment_funny',
       'compliment_writer', 'compliment_photos'],
      dtype='object')

In [19]:
selected_columns_train_reviews =  full_train_reviews[learnable_columns]
selected_columns_test_reviews =  full_test_reviews[learnable_columns]

Dividimos el dataset en train y test

In [20]:
# dividimos X e y

from sklearn.model_selection import train_test_split

y = full_train_reviews['stars_review']
X = selected_columns_train_reviews

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y
)

### Utilizamos un tree regresor

In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix

model = DecisionTreeRegressor(random_state=SEED)

param_grid = {
    'max_depth': [10, 20,40],
    'min_samples_split': [10, 50, 100],
    'min_samples_leaf': [5, 20, 50],
    'criterion': ['squared_error']
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='f1_weighted', 
    cv=2,
    n_jobs=4,  # parallel execution
    verbose=2
)

grid_search.fit(X_train, y_train)

print(f"best params {grid_search.best_params_}")


Fitting 2 folds for each of 27 candidates, totalling 54 fits


c:\Users\pable\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:976: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan]
  warnings.warn(


best params {'criterion': 'squared_error', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 10}


In [22]:
# posteriormente vemos que tal predice
from sklearn.metrics import mean_absolute_error

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"absolute error {mean_absolute_error(y_test, y_pred)}")

absolute error 0.7527752303960105


Unicamente usando datos simples numericos del dataset y con un arbol bastante simple obtenemos un RMSE mejor que el dummy, sin embargo existe bastante margen de mejora

In [23]:
# Observamos la importancia de las variables
importance = best_model.feature_importances_
feature_names = X_train.columns

# Create DataFrame
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

# Display top 20 features
print(feature_importance.head(20))

              Feature  Importance
4       average_stars    0.618027
18     stars_business    0.292168
0              useful    0.027952
2                cool    0.026244
1               funny    0.020028
22        useful_user    0.003256
24          cool_user    0.002602
20  review_count_user    0.002495
23         funny_user    0.001922
11   compliment_plain    0.000956
14  compliment_writer    0.000931
17          longitude    0.000776
19       review_count    0.000774
16           latitude    0.000402
3                fans    0.000317
5      compliment_hot    0.000235
12    compliment_cool    0.000202
6     compliment_more    0.000157
13   compliment_funny    0.000126
21            is_open    0.000104


Podemos ver que el modelo esta utilizando sobre todo la media de las estrellas del usuario y del local para realizar las predicciones lo cual es esperado ya que aportan informacion muy importante sobre la valoracion que da el usuario y la valoracion media del restaurante. Sin embargo es curioso observar como tambien tiene en cuenta las votaciones de gracioso, util y guay que la gente le da a la review

### Utilizamos un XGBoost

In [24]:
from xgboost import XGBRegressor

# XGBoost Regressor instance
xgb = XGBRegressor(
    objective='reg:squarederror',
    n_jobs=-1,
    verbosity=1,
    tree_method='hist'  # Optimized for large datasets
)

# Grid search parameters (keep small to start — expand if needed)
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [6, 10],
    'learning_rate': [0.1, 0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Grid search with 3-fold CV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2
)

# Fit model
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Evaluate
y_pred = best_model.predict(X_test)
abserr = mean_absolute_error(y_test, y_pred)

# Output
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test absolute error: {abserr:.4f}")

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   2.1s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   1.8s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   1.9s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time=   3.3s
[CV] END colsample_bytree=0.8, learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time=   4.5s
[CV] END colsample_

Utilizando un xgboost se obtienen valores algo mejores que con el arbol de regresion, probamos a realizar un test con los datos de test de kaggle.

In [25]:
submition = pd.DataFrame({'review_id': full_test_reviews['review_id']})
submition['stars'] = best_model.predict(selected_columns_test_reviews.to_numpy())

submition.to_csv('./submission_simple_xgboost.csv', index=False)

En este test se obtienen valores de 0.7293, que son mucho mejores que lo obtenido en el test. Esto puede ser debido a que en el test de kaggle se utilice una metrica de error diferente.

### Utilizando mas variables

Existen varias variables que no se han utilizado en los modelos de ML, que no requieren informacion semantica del texto, cosas como el estado o ciudad en la que se encuentra el restaurante puede ser bastante valiosa. Ademas variables con muchos valores como los atributos del restaurante o sus categorias tambien pueden aportar mucha informacion.

En primer lugar transformamos la informacion del estado y la ciudad donde se encuentra el restaurante

In [26]:
from sklearn.preprocessing import LabelEncoder
business_data_encoded = business_data.copy()

business_data_encoded['city'] = LabelEncoder().fit_transform(business_data['city'])
business_data_encoded['state'] = LabelEncoder().fit_transform(business_data['state'])
business_data_encoded[['city','state']]

,city,state
0,335,5
1,397,9
2,560,5
3,646,5
4,646,5
...,...,...
30064,603,5
30065,558,10
30066,87,8
30067,116,5


Ponemos la variable categories que indica las categorias del restaurante como variable categorica

In [27]:
categorie_data = []
nan_count = 0
for row in business_data['categories']:
    if isinstance(row, str):
        categorie_data.append(row.split(','))
    else:
        nan_count += 1
print(f"numero de filas con nans = {nan_count}")
print(len(categorie_data))

numero de filas con nans = 17
30052


In [28]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
categories_encoded = mlb.fit_transform(categorie_data)
categories_df = pd.DataFrame(categories_encoded, columns=mlb.classes_)
print(categories_df.shape)
categories_df.head()

(30052, 2113)


,& Probates,3D Printing,ATV Rentals/Tours,Acai Bowls,Accessories,Accountants,Acne Treatment,Active Life,Acupuncture,Addiction Medicine,...,Wine Bars,Wine Tasting Room,Wine Tours,Wineries,Women's Clothing,Workers Compensation Law,Wraps,Yelp Events,Yoga,Zoos
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Tras realizar esta operacion vemos que unicamente añadiriamos 2113 columnas lo cual es una barbaridad pero podria aportar informacion importante.

In [29]:
# añadimos al dataframe de business los datos de los atributos

business_data_encoded['categories'] = business_data['categories'].astype(str).fillna('')

# Split categories and transform using MultiLabelBinarizer
category_lists = business_data_encoded['categories'].apply(lambda x: [cat.strip() for cat in x.split(',') if cat.strip()])
labels = mlb.transform(category_lists)

# Create DataFrame from labels
categories = pd.DataFrame(labels, columns=mlb.classes_)
categories['business_id'] = business_data_encoded['business_id'].values

# concatenate both dataframe

business_data_encoded = pd.merge(business_data_encoded, categories, on='business_id', how='left')
business_data_encoded.head()

c:\Users\pable\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:900: UserWarning: unknown class(es) ['& Probates', '3D Printing', 'ATV Rentals/Tours', 'Acne Treatment', 'Addiction Medicine', 'Aerial Tours', 'Aestheticians', 'Aircraft Repairs', 'Airport Terminals', 'Anesthesiologists', 'Apartment Agents', 'Armenian', 'Art Consultants', 'Art Installation', 'Art Space Rentals', 'Artificial Turf', 'Audio/Visual Equipment Rental', 'Audiologist', 'Australian', 'Austrian', 'Awnings', 'Ayurveda', 'Backflow Services', 'Balloon Services', 'Bangladeshi', 'Bar Crawl', 'Baseball Fields', 'Batting Cages', 'Beach Bars', 'Beach Equipment Rentals', 'Behavior Analysts', 'Beverage Store', 'Bike Sharing', 'Bike tours', 'Biohazard Cleanup', 'Bistros', 'Boat Parts & Supplies', 'Boat Tours', 'Bookbinding', 'Boudoir Photography', 'Bounce House Rentals', 'Bus Tours', 'Business Financing', 'CPR Classes', 'Calligraphy', 'Cannabis Collective', 'Car Brokers', 'Caricatures', 'Cheerleading', 'Cigar Bars',

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,Wine Bars,Wine Tasting Room,Wine Tours,Wineries,Women's Clothing,Workers Compensation Law,Wraps,Yelp Events,Yoga,Zoos
0,GDEEPQdYs2utMN-R4znZSA,Metro Self Storage - Largo,10501 S Belcher Rd,335,5,33777,27.868519,-82.743849,4.5,7,...,0,0,0,0,0,0,0,0,0,0
1,pbAq2NRG_2jCBI6fgRalvQ,Madewell,"3301 Veterans Blvd, Ste 84",397,9,70002,30.005894,-90.157450,2.0,6,...,0,0,0,0,1,0,0,0,0,0
2,h4HP3Vc0dQq7SfSLal9qQw,Dollylocks,511 9th St N,560,5,33701,27.777785,-82.646673,4.5,7,...,0,0,0,0,0,0,0,0,0,0
3,PndbFVbHE4730HDlghxv6g,Jim Browne Chrysler Jeep Dodge Ram of Tampa Bay,10909 N Florida Ave,646,5,33612,28.048093,-82.458757,2.5,76,...,0,0,0,0,0,0,0,0,0,0
4,IayDnngl0NooAbcoo62j-w,Emg Salons,324 S Falkenburg Rd,646,5,33619,27.948815,-82.334619,4.5,7,...,0,0,0,0,0,0,0,0,0,0


unimos los datos de busines a todo el dataset

In [30]:
full_train_reviews = pd.merge(train_reviews, business_data_encoded, on='business_id', how='left',suffixes=('_review', '_business'))
full_train_reviews = pd.merge(full_train_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = pd.merge(test_reviews, business_data_encoded, on='business_id', how='left',suffixes=('_review', '_business'))
full_test_reviews = pd.merge(full_test_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = full_test_reviews.rename(columns={"stars": "stars_business"})
full_test_reviews.head()

,review_id,user_id,business_id,useful,funny,cool,text,date,name,address,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,Backatown Coffee Parlour,"301 Basin St, Ste 1",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,Lincoln Square Pancake House,5024 E 56th St,...,6.0,1.0,0.0,3.0,16.0,28.0,32.0,32.0,18.0,1.0
2,seR2KhblYMWg-k9zzN6aYA,hF68a0mpu97u0oaryFYhyg,_RG4IByyBR528CMc7DefJA,2,0,0,Came here when my kitten got very sick by the ...,2015-09-06 15:29:02,BluePearl Pet Hospital,301 Veterans Hwy,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,BToo00Fi5pfJFA5MI2HM5g,G4yX5Q1tFfwSucFOmiyjdA,xxlbRiWWQkk-6LST3Hd12g,2,0,0,So I'll preface by saying we did have an overa...,2015-09-14 00:49:17,The Backyard,244 W Harrison Ave,...,4.0,0.0,0.0,0.0,11.0,6.0,10.0,10.0,6.0,1.0
4,FHJAzi1imodBit3RWK7zQA,Srqi1xb7exdB9uRHxDeEkw,LgGqdFLD7-ca0Z9F_q4Fuw,0,0,0,This place is a joke. Worst bar service ever. ...,2015-07-24 01:03:40,Postcard Inn on the Beach,6300 Gulf Blvd,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [31]:
learnable_columns = ['useful','funny','cool','fans','average_stars','compliment_hot','compliment_more','compliment_profile','compliment_cute','compliment_list',
                     'compliment_note','compliment_plain','compliment_cool','compliment_funny','compliment_writer','compliment_photos','latitude','longitude',
                     'stars_business','review_count','review_count_user','is_open','useful_user','funny_user','cool_user']

learnable_columns = learnable_columns + ['city','state']
learnable_columns = learnable_columns + mlb.classes_.tolist()
selected_columns_train_reviews =  full_train_reviews[learnable_columns]
selected_columns_test_reviews =  full_test_reviews[learnable_columns]

In [32]:
from sklearn.model_selection import train_test_split
y = full_train_reviews['stars_review']
X = selected_columns_train_reviews

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y
)

Utilizamos un arbol de decision para las predicciones

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix

model = DecisionTreeRegressor(random_state=SEED)

param_grid = {
    'max_depth': [10, 30],
    'min_samples_split': [10, 30],
    'min_samples_leaf': [5, 20],
    'criterion': ['squared_error']
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='f1_weighted', 
    cv=2,
    n_jobs=4,  # parallel execution
    verbose=2
)

grid_search.fit(X_train, y_train)

print(f"best params {grid_search.best_params_}")

# posteriormente vemos que tal predice
from sklearn.metrics import mean_absolute_error

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"absolute error {mean_absolute_error(y_test, y_pred)}")

Fitting 2 folds for each of 8 candidates, totalling 16 fits


c:\Users\pable\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:976: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


best params {'criterion': 'squared_error', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 10}
absolute error 0.7536637548738971


In [34]:
# Observamos la importancia de las variables
importance = best_model.feature_importances_
feature_names = X_train.columns

# Create DataFrame
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

# Display top 20 features
print(feature_importance.head(20))

                Feature  Importance
4         average_stars    0.617567
18       stars_business    0.292084
0                useful    0.027913
2                  cool    0.026123
1                 funny    0.019975
22          useful_user    0.003110
24            cool_user    0.002608
20    review_count_user    0.002352
23           funny_user    0.001851
11     compliment_plain    0.001019
14    compliment_writer    0.000822
19         review_count    0.000816
17            longitude    0.000585
25                 city    0.000328
3                  fans    0.000280
16             latitude    0.000258
5        compliment_hot    0.000245
1356        Car Dealers    0.000203
1924        Restaurants    0.000183
12      compliment_cool    0.000161


Al observar la importancia de las variables, vemos que el modelo no esta usando las nuevas variables.

Utilizamos un XGBoost a ver si aprende a utilizar las nuevas variables

In [35]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
# XGBoost Regressor instance
xgb = XGBRegressor(
    objective='reg:squarederror',
    n_jobs=-1,
    verbosity=1,
    tree_method='hist'  # Optimized for large datasets
)

# Grid search parameters (keep small to start — expand if needed)
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [6, 10],
    'learning_rate': [0.1, 0.05],
    'subsample': [0.8, 1.0]
}

# Grid search with 3-fold CV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=2,
    verbose=2
)

# Fit model
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_



Fitting 2 folds for each of 16 candidates, totalling 32 fits
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=  41.7s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time= 1.2min
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=  38.2s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time= 1.2min
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time= 1.1min
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time= 1.8min
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=1.0; total time= 1.1min
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=1.0; total time= 1.7min
[CV] END learning_rate=0.1, max_depth=10, n_estimators=100, subsample=0.8; total time=  45.7s
[CV] END learning_rate=0.1, max_depth=10, n_estimators=100, subsample=0.8; total time= 1.3min
[CV] EN

In [36]:
from sklearn.metrics import mean_absolute_error
# Evaluate
y_pred = best_model.predict(X_test)
abserr = mean_absolute_error(y_test, y_pred)

# Output
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test absolute error: {abserr:.4f}")


Best Parameters: {'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 300, 'subsample': 0.8}
Test absolute error: 0.7262


Se obtienen valores algo mejores que utilizando menos variables obteniendo un RMSE de 1.0076, probamos a utilizarlo en kaggle.

In [37]:
submition = pd.DataFrame({'review_id': full_test_reviews['review_id']})
submition['stars'] = best_model.predict(selected_columns_test_reviews.to_numpy())

submition.to_csv('./submission_full_vars_xgboost.csv', index=False)

 En el test de kaggle se obtienen valores de 0.7266 que son algo superiores a usar menos variables, donde se obtiene un RMSE de 0.7293, sin embargo no es una mejora significativa.

## Probamos a ver si en la variable atributes existen valores importantes para que el modelo pueda aprender

convertimos la columna a un json y lo analizamos

In [38]:
import ast

# convertimos a un json valido
json_atr_column = business_data['attributes'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
json_atr_column

0        {'BusinessAcceptsCreditCards': 'True', 'Wheelc...
1                   {'BusinessAcceptsCreditCards': 'True'}
2        {'RestaurantsPriceRange2': '4', 'GoodForKids':...
3                   {'BusinessAcceptsCreditCards': 'True'}
4        {'WiFi': 'u'free'', 'BikeParking': 'True', 'Bu...
                               ...                        
30064    {'Ambience': '{'touristy': False, 'hipster': F...
30065    {'ByAppointmentOnly': 'False', 'BusinessAccept...
30066    {'BusinessAcceptsCreditCards': 'True', 'ByAppo...
30067    {'BikeParking': 'True', 'BusinessAcceptsCredit...
30068                                                  NaN
Name: attributes, Length: 30069, dtype: object

In [39]:
from collections import defaultdict, Counter

value_collector = defaultdict(Counter)

def collect_values(obj, prefix=''):
    if isinstance(obj, dict):
        for k, v in obj.items():
            full_key = f"{prefix}.{k}" if prefix else k
            collect_values(v, full_key)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            collect_values(item, f"{prefix}[{i}]")
    else:
        value_collector[prefix][str(obj)] += 1

for json_obj in json_atr_column:
    collect_values(json_obj)

# Example: show top 5 values for each key
for k, counter in value_collector.items():
    print(f"{k}: {counter.most_common(5)}")

BusinessAcceptsCreditCards: [('True', 22643), ('False', 1237), ('None', 8)]
WheelchairAccessible: [('True', 5119), ('False', 603), ('None', 8)]
RestaurantsPriceRange2: [('2', 9593), ('1', 5827), ('3', 1377), ('4', 233), ('None', 5)]
GoodForKids: [('True', 8869), ('False', 1854), ('None', 8)]
BusinessAcceptsBitcoin: [('False', 3315), ('True', 91)]
ByAppointmentOnly: [('False', 5404), ('True', 3093), ('None', 9)]
BikeParking: [('True', 10892), ('False', 3543), ('None', 18)]
BusinessParking: [("{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 6793), ("{'garage': False, 'street': False, 'validated': False, 'lot': False, 'valet': False}", 4829), ("{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}", 2645), ("{'garage': False, 'street': True, 'validated': False, 'lot': True, 'valet': False}", 1071), ('None', 447)]
HairSpecializesIn: [("{'straightperms': False, 'coloring': False, 'extensions': False, 'africanamerican': Fals

Dado que algunas filas son jsons y nested jsons, los aplanamos y añadimos como columnas.

In [40]:
def flatten_dict(d, parent_key='', sep='.'):
    items = []
    if isinstance(d, float):
        return dict(items)
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        # Try to parse stringified dicts
        if isinstance(v, str):
            try:
                v_parsed = ast.literal_eval(v)
                if isinstance(v_parsed, dict):
                    v = v_parsed
            except (ValueError, SyntaxError):
                pass  # Not a dict, keep as string

        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


flat_atr_data = business_data['attributes'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
flat_atr_data = flat_atr_data.apply(flatten_dict)
flat_df = pd.DataFrame(flat_atr_data.tolist())
flat_df.head()

,BusinessAcceptsCreditCards,WheelchairAccessible,RestaurantsPriceRange2,GoodForKids,BusinessAcceptsBitcoin,ByAppointmentOnly,BikeParking,BusinessParking.garage,BusinessParking.street,BusinessParking.validated,...,Music,BestNights,HairSpecializesIn,DietaryRestrictions.dairy-free,DietaryRestrictions.gluten-free,DietaryRestrictions.vegan,DietaryRestrictions.kosher,DietaryRestrictions.halal,DietaryRestrictions.soy-free,DietaryRestrictions.vegetarian
0,True,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,False,4,False,False,False,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,True,True,2,True,NaN,True,True,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
print(flat_df.shape)
flat_df.describe()

(30069, 87)


,BusinessAcceptsCreditCards,WheelchairAccessible,RestaurantsPriceRange2,GoodForKids,BusinessAcceptsBitcoin,ByAppointmentOnly,BikeParking,BusinessParking.garage,BusinessParking.street,BusinessParking.validated,...,Music,BestNights,HairSpecializesIn,DietaryRestrictions.dairy-free,DietaryRestrictions.gluten-free,DietaryRestrictions.vegan,DietaryRestrictions.kosher,DietaryRestrictions.halal,DietaryRestrictions.soy-free,DietaryRestrictions.vegetarian
count,23888,5730,17035,10731,3406,8506,14453,17354,17065,17320,...,6,3,6,4,4,4,4,4,4,4
unique,3,3,5,3,2,3,3,2,2,2,...,1,1,1,2,2,2,1,1,2,2
top,True,True,2,True,False,False,True,False,False,False,...,None,None,None,True,True,True,False,False,True,True
freq,22643,5119,9593,8869,3315,5404,10892,16522,12492,17150,...,6,3,6,2,3,2,4,4,2,3


In [42]:
# porcentaje de valores null en las nuevas columnas
print(f"media de porcentajes de valores null en las columnas {(flat_df.isna().sum(axis = 0)/len(flat_df)).mean()}")
print(f"numero de columnas cuyo porcentaje de valores faltantes es menor del 60%: {(flat_df.isna().sum(axis = 0)/len(flat_df)< 0.6).sum()}")

media de porcentajes de valores null en las columnas 0.8422574438943687
numero de columnas cuyo porcentaje de valores faltantes es menor del 60%: 8


Podemos observar que existen muchas variables que tienen un gran porcentaje de valores nulos (de media mas del 80% de los valores son nulos). Tan solo 8 columnas tienen mas del 40% de datos que no son nulos. Observamos esas columnas

In [43]:
flat_df.loc[:,(flat_df.isna().sum(axis = 0)/len(flat_df)< 0.6)]

,BusinessAcceptsCreditCards,RestaurantsPriceRange2,BikeParking,BusinessParking.garage,BusinessParking.street,BusinessParking.validated,BusinessParking.lot,BusinessParking.valet
0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,4,False,False,False,False,False,False
3,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,True,2,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...
30064,NaN,NaN,NaN,False,None,False,True,False
30065,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30066,True,2,True,False,False,False,False,False
30067,True,NaN,True,False,False,False,True,False


Las columnas que conservan los suficientes valores no nulos como para que puedan considerarse importantes son: BusinessAcceptsCreditCards	RestaurantsPriceRange2	BikeParking	BusinessParking.garage	BusinessParking.street	BusinessParking.validated	BusinessParking.lot y BusinessParking.valet. Donde todos son valores booleanos y solo RestaurantsPriceRange2 es una variable numerica. 

Vamos a añadir estas variables para entrenar el modelo.

In [44]:
flat_df_good_cols = flat_df.loc[:,(flat_df.isna().sum(axis = 0)/len(flat_df)< 0.6)]

In [45]:
# cambiamos el tipo de las columnas para que sean boleanos y enteros en vez de strings
from sklearn.impute import SimpleImputer

cols_to_convert = [
    'BusinessAcceptsCreditCards',
    'BikeParking',
    'BusinessParking.garage',
    'BusinessParking.street',
    'BusinessParking.validated',
    'BusinessParking.lot',
    'BusinessParking.valet'
]

flat_df_good_cols[cols_to_convert] = flat_df_good_cols[cols_to_convert].applymap(lambda x: str(x).lower() == 'true' if pd.notnull(x) else False)
flat_df_good_cols[cols_to_convert] = flat_df_good_cols[cols_to_convert].applymap(lambda x: bool(int(x)) if pd.notnull(x) else False)

flat_df_good_cols[cols_to_convert] = flat_df_good_cols[cols_to_convert].astype('boolean')

flat_df_good_cols['RestaurantsPriceRange2'] = flat_df_good_cols['RestaurantsPriceRange2'].replace(['None', 'none', 'NULL', '', 'NaN'], np.nan)
flat_df_good_cols['RestaurantsPriceRange2'] = flat_df_good_cols['RestaurantsPriceRange2'].astype('float')

# transformamos los valores por la media
imputer = SimpleImputer(strategy='mean')  # or 'median', 'most_frequent'
flat_df_good_cols['RestaurantsPriceRange2'] = imputer.fit_transform(flat_df_good_cols[['RestaurantsPriceRange2']])


C:\Users\pable\AppData\Local\Temp\ipykernel_2732\1052318792.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flat_df_good_cols[cols_to_convert] = flat_df_good_cols[cols_to_convert].applymap(lambda x: str(x).lower() == 'true' if pd.notnull(x) else False)
C:\Users\pable\AppData\Local\Temp\ipykernel_2732\1052318792.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flat_df_good_cols[cols_to_convert] = flat_df_good_cols[cols_to_convert].applymap(lambda x: bool(int(x)) if pd.notnull(x) else False)
C:\Users

Transformamos las variables para que puedan ser utilizadas por el modelo

In [46]:
# Concatenamos los dataframes
business_data_atributes = pd.concat([business_data, flat_df_good_cols], axis=1)

full_train_reviews = pd.merge(train_reviews, business_data_atributes, on='business_id', how='left',suffixes=('_review', '_business'))
full_train_reviews = pd.merge(full_train_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = pd.merge(test_reviews, business_data_atributes, on='business_id', how='left',suffixes=('_review', '_business'))
full_test_reviews = pd.merge(full_test_reviews, user_data, on='user_id', how='left',suffixes=(None, '_user'))

full_test_reviews = full_test_reviews.rename(columns={"stars": "stars_business"})
full_test_reviews.head()

,review_id,user_id,business_id,useful,funny,cool,text,date,name,address,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,ieYPmCImINjPzTDFmEKBKA,79F9QrQSet-b1yRCIM243Q,sXSUzImYOcRRI3xtG2M85g,1,0,1,Amazing coffee and chill atmosphere. The staff...,2018-01-29 04:33:28,Backatown Coffee Parlour,"301 Basin St, Ste 1",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,QIkJ8fZ4yx_QaHahWWszAA,chuM6TBkFHtTwJ6z96Hj1A,Ipt9ga67vVC_2ob3YmVwNA,4,0,2,I pass by this joint every time I make a run t...,2011-01-10 03:10:49,Lincoln Square Pancake House,5024 E 56th St,...,6.0,1.0,0.0,3.0,16.0,28.0,32.0,32.0,18.0,1.0
2,seR2KhblYMWg-k9zzN6aYA,hF68a0mpu97u0oaryFYhyg,_RG4IByyBR528CMc7DefJA,2,0,0,Came here when my kitten got very sick by the ...,2015-09-06 15:29:02,BluePearl Pet Hospital,301 Veterans Hwy,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,BToo00Fi5pfJFA5MI2HM5g,G4yX5Q1tFfwSucFOmiyjdA,xxlbRiWWQkk-6LST3Hd12g,2,0,0,So I'll preface by saying we did have an overa...,2015-09-14 00:49:17,The Backyard,244 W Harrison Ave,...,4.0,0.0,0.0,0.0,11.0,6.0,10.0,10.0,6.0,1.0
4,FHJAzi1imodBit3RWK7zQA,Srqi1xb7exdB9uRHxDeEkw,LgGqdFLD7-ca0Z9F_q4Fuw,0,0,0,This place is a joke. Worst bar service ever. ...,2015-07-24 01:03:40,Postcard Inn on the Beach,6300 Gulf Blvd,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [47]:
from sklearn.model_selection import train_test_split

learnable_columns = ['useful','funny','cool','fans','average_stars','compliment_hot','compliment_more','compliment_profile','compliment_cute','compliment_list',
                     'compliment_note','compliment_plain','compliment_cool','compliment_funny','compliment_writer','compliment_photos','latitude','longitude',
                     'stars_business','review_count','review_count_user','is_open','useful_user','funny_user','cool_user']

learnable_columns = learnable_columns + flat_df_good_cols.columns.tolist()
selected_columns_train_reviews =  full_train_reviews[learnable_columns]
selected_columns_test_reviews =  full_test_reviews[learnable_columns]


y = full_train_reviews['stars_review']
X = selected_columns_train_reviews

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y
)

Utilizamos un arbol de decision para realizar las predicciones

In [48]:
from sklearn.model_selection import GridSearchCV
import scipy.sparse.linalg
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix

model = DecisionTreeRegressor(random_state=SEED)

param_grid = {
    'max_depth': [5, 10, 20, 30,40],
    'min_samples_split': [5, 10, 50, 100],
    'min_samples_leaf': [3, 5, 20, 50,70],
    'criterion': ['squared_error']
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='f1_weighted', 
    cv=3,
    n_jobs=4,  # parallel execution
    verbose=2
)

grid_search.fit(X_train, y_train)

print(f"best params {grid_search.best_params_}")

# posteriormente vemos que tal predice
from sklearn.metrics import mean_absolute_error

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
print(f"absolute error {mean_absolute_error(y_test, y_pred)}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits


c:\Users\pable\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:976: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


best params {'criterion': 'squared_error', 'max_depth': 5, 'min_samples_leaf': 3, 'min_samples_split': 5}
absolute error 0.807973343242946


In [49]:
# Observamos la importancia de las variables
importance = best_model.feature_importances_
feature_names = X_train.columns

# Create DataFrame
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

# Display top 20 features
print(feature_importance.head(20))

                       Feature  Importance
4                average_stars    0.674865
18              stars_business    0.314488
1                        funny    0.006444
0                       useful    0.004203
30   BusinessParking.validated    0.000000
29      BusinessParking.street    0.000000
28      BusinessParking.garage    0.000000
27                 BikeParking    0.000000
31         BusinessParking.lot    0.000000
25  BusinessAcceptsCreditCards    0.000000
24                   cool_user    0.000000
23                  funny_user    0.000000
22                 useful_user    0.000000
21                     is_open    0.000000
20           review_count_user    0.000000
19                review_count    0.000000
26      RestaurantsPriceRange2    0.000000
16                    latitude    0.000000
17                   longitude    0.000000
15           compliment_photos    0.000000


Parece que el arbol de decision no utiliza las nuevas variables insertadas, ya que no mejora el error y no les da importancia a las nuevas variables. Esto puede sugerir que no son utiles para la clasificacion o que el modelo no ha aprendido a usarlas.

Utilizando un XGBoost

In [50]:
from xgboost import XGBRegressor

# XGBoost Regressor instance
xgb = XGBRegressor(
    objective='reg:squarederror',
    n_jobs=-1,
    verbosity=1,
    tree_method='hist'  # Optimized for large datasets
)

# Grid search parameters (keep small to start — expand if needed)
param_grid = {
    'n_estimators': [100, 300,500],
    'max_depth': [6, 10],
    'learning_rate': [0.1, 0.05],
    'subsample': [0.8, 1.0]
}

# Grid search with 3-fold CV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2
)

# Fit model
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Evaluate
y_pred = best_model.predict(X_test)
abserr = mean_absolute_error(y_test, y_pred)

# Output
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test absolute error: {abserr:.4f}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   1.7s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   2.2s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=0.8; total time=   2.2s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   1.4s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   2.0s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=100, subsample=1.0; total time=   1.9s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time=   3.7s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time=   5.1s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=0.8; total time=   5.0s
[CV] END learning_rate=0.1, max_depth=6, n_estimators=300, subsample=1.0; total time=   2.9s
[CV] END 

In [51]:
submition = pd.DataFrame({'review_id': full_test_reviews['review_id']})
submition['stars'] = best_model.predict(selected_columns_test_reviews.to_numpy())

submition.to_csv('./submission_atr_vars_xgboost.csv', index=False)

Realizamos las predicciones sobre test y las probamos en kaggle, obteniendo un error de 0.7261, que es un poco superior a los resultados de no utilizar la informacion de la variable atributes de 0.7293. Esto sugiere que la informacion de esta variable no aporta informacion util para realizar la clasificacion o que se puede inferir en base a las demas variables.

Probamos a utilizar todo el dataset disponible para intentar obtener los mejores resultados usando este modelo.

In [52]:
# Define model with given hyperparameters
xgb_model = XGBRegressor(
    colsample_bytree=1.0,
    learning_rate=0.05,
    max_depth=10,
    n_estimators=300,
    subsample=0.8,
    objective='reg:squarederror',  # explicitly set for regression
    n_jobs=-1,                     # utilize all CPU cores
    verbosity=1                    # set to 0 to silence output
)

# Fit the model
xgb_model.fit(X, y)
submition = pd.DataFrame({'review_id': full_test_reviews['review_id']})
submition['stars'] = best_model.predict(selected_columns_test_reviews.to_numpy())
submition.to_csv('./submission_atr_vars_xgboost_all_rows.csv', index=False)